# UrbanPulse Nepal — Data Audit and Preprocessing

This notebook starts from the BIPAD incident snapshot created in notebook 01. It documents the observed retrieval results, audits important fields, normalizes incident dates and GeoJSON points, derives calendar features, and saves a processed dataset and quality report for later analysis.

## Findings from the saved BIPAD retrieval

- **64,735 incident records** were received across 65 non-empty API pages; page 65 contained 735 records and page 66 was empty.
- Every record had a non-missing, unique `id` in the saved CSV.
- `incidentOn` spans **2011-04-14 through 2026-10-13**; `createdOn` spans 2019-04-09 through 2026-10-13.
- The API returned `count = 9,223,372,036,854,775,807` on pages. This is not a credible total, so it is treated as an invalid/sentinel count.
- The retrieval notebook marked pagination incomplete because it stopped on an empty page. The observed coverage is therefore **not verified as a complete historical archive**.
- Returned hazard values are numeric IDs (20 distinct in this snapshot), not readable labels. This notebook preserves those IDs rather than guessing their meanings.

These results describe the saved retrieval only. They do not establish completeness of BIPAD’s historical records. The raw files remain unchanged.

## 1. Load the saved snapshot and inspect its schema

The notebook selects the newest folder containing both `records.json` and `retrieval_manifest.json`. If you want to analyze a specific prior retrieval, set `RUN_DIR` explicitly in the configuration cell.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import ast, json, math
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').is_dir(): PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'data').is_dir(): raise FileNotFoundError('Run from the repository or notebooks/ folder.')
RAW_ROOT = PROJECT_ROOT / 'data' / 'raw' / 'bipad_incidents'
candidates = sorted((p for p in RAW_ROOT.iterdir() if (p / 'records.json').is_file() and (p / 'retrieval_manifest.json').is_file()), key=lambda p:p.name)
if not candidates: raise FileNotFoundError(f'No completed notebook 01 output found under {RAW_ROOT}')
RUN_DIR = candidates[-1]  # Change this to a specific run folder if needed.
manifest = json.loads((RUN_DIR / 'retrieval_manifest.json').read_text(encoding='utf-8'))
records = json.loads((RUN_DIR / 'records.json').read_text(encoding='utf-8'))
raw_df = pd.DataFrame.from_records(records)
print('Input snapshot:', RUN_DIR)
print('Manifest summary:', {k:manifest.get(k) for k in ['pages_saved','records_received','api_reported_total','pagination_complete','stop_reason']})
print('Shape:', raw_df.shape)
print('Fields:', sorted(raw_df.columns.tolist()))
display(raw_df.head(5))

## 2. Audit identifiers, timestamps, hazard IDs, and locations

The API represents a point as GeoJSON coordinates in `[longitude, latitude]` order. We retain all input rows and add flags so questionable coordinates remain traceable instead of silently disappearing. Dates with offsets are parsed and converted to Nepal time for calendar features. The hazard ID remains a category until it can be joined to a verified BIPAD hazard lookup.

In [ ]:
def extract_coordinates(value):
    # CSV exports may represent nested values as Python-style strings; records.json usually supplies objects.
    if isinstance(value, str):
        try: value = ast.literal_eval(value)
        except (ValueError, SyntaxError): return (float('nan'), float('nan'))
    if not isinstance(value, dict): return (float('nan'), float('nan'))
    coords = value.get('coordinates')
    if not isinstance(coords, (list, tuple)) or len(coords) < 2: return (float('nan'), float('nan'))
    try: return (float(coords[0]), float(coords[1]))  # GeoJSON order is longitude, latitude.
    except (TypeError, ValueError): return (float('nan'), float('nan'))

df = raw_df.copy()
if 'id' in df.columns:
    df['id'] = df['id'].astype('string')
    duplicate_id_mask = df['id'].duplicated(keep='first') & df['id'].notna()
else:
    duplicate_id_mask = pd.Series(False, index=df.index)
    df['id'] = pd.Series(pd.NA, index=df.index, dtype='string')
duplicate_id_count = int(duplicate_id_mask.sum())
missing_id_count = int(df['id'].isna().sum())
# Retain the first occurrence of duplicated identifiers and report how many were removed.
df = df.loc[~duplicate_id_mask].copy()

for col in ['incidentOn','createdOn','reportedOn','modifiedOn']:
    if col in df.columns:
        parsed = pd.to_datetime(df[col], errors='coerce', utc=True)
        df[f'{col}_nepal'] = parsed.dt.tz_convert('Asia/Kathmandu')

coords = df['point'].map(extract_coordinates) if 'point' in df.columns else pd.Series([(float('nan'),float('nan'))]*len(df), index=df.index)
df['longitude'] = coords.map(lambda pair:pair[0])
df['latitude'] = coords.map(lambda pair:pair[1])
numeric_coordinates = df['longitude'].map(math.isfinite) & df['latitude'].map(math.isfinite)
df['coordinates_present'] = numeric_coordinates
df['coordinates_in_broad_nepal_bounds'] = numeric_coordinates & df['latitude'].between(26,31) & df['longitude'].between(80,89)

if 'hazard' in df.columns: df['hazard_id'] = df['hazard'].astype('string')
else: df['hazard_id'] = pd.Series(pd.NA, index=df.index, dtype='string')
if 'incidentOn_nepal' in df.columns:
    df['incident_year'] = df['incidentOn_nepal'].dt.year.astype('Int64')
    df['incident_month'] = df['incidentOn_nepal'].dt.month.astype('Int64')
    df['incident_season'] = df['incident_month'].map({12:'Winter',1:'Winter',2:'Winter',3:'Pre-monsoon',4:'Pre-monsoon',5:'Pre-monsoon',6:'Monsoon',7:'Monsoon',8:'Monsoon',9:'Post-monsoon',10:'Post-monsoon',11:'Post-monsoon'}).astype('string')

audit = {'snapshot_folder':RUN_DIR.name, 'retrieved_at_utc':manifest.get('retrieved_at_utc'), 'input_rows':len(raw_df), 'processed_rows':len(df), 'duplicate_ids_removed':duplicate_id_count, 'missing_ids':missing_id_count, 'api_reported_total':manifest.get('api_reported_total'), 'pagination_complete':manifest.get('pagination_complete'), 'retrieval_stop_reason':manifest.get('stop_reason'), 'missing_or_invalid_coordinates':int((~df['coordinates_present']).sum()), 'coordinates_outside_broad_nepal_bounds':int((df['coordinates_present'] & ~df['coordinates_in_broad_nepal_bounds']).sum()), 'hazard_id_count':int(df['hazard_id'].nunique(dropna=True))}
for source_col, output_key in [('incidentOn_nepal','incident_date'),('createdOn_nepal','created_date')]:
    if source_col in df.columns:
        values = df[source_col].dropna()
        audit[f'{output_key}_missing_or_unparseable'] = int(df[source_col].isna().sum())
        audit[f'{output_key}_min'] = values.min().isoformat() if len(values) else None
        audit[f'{output_key}_max'] = values.max().isoformat() if len(values) else None
display(pd.Series(audit, name='value').to_frame())

## 3. Explore temporal and hazard distributions

These tables describe the retrieved records, not population-level disaster risk. Yearly counts can be affected by changes in reporting, coverage, and data-entry practices. Hazard groups are shown by API ID because verified display labels are not included in the saved incident response.

In [ ]:
if 'incident_year' in df.columns:
    yearly_counts = df.groupby('incident_year', dropna=False).size().rename('incident_records').reset_index()
    display(yearly_counts.tail(20))
    ax = yearly_counts.dropna(subset=['incident_year']).plot(x='incident_year', y='incident_records', kind='line', marker='o', figsize=(11,4), legend=False, title='BIPAD incident records by incident year')
    ax.set_xlabel('Incident year'); ax.set_ylabel('Records in retrieved snapshot'); ax.grid(alpha=0.25)

hazard_counts = df.groupby('hazard_id', dropna=False).size().sort_values(ascending=False).rename('incident_records').reset_index()
display(hazard_counts.head(25))
if 'dataSource' in df.columns:
    display(df['dataSource'].value_counts(dropna=False).rename_axis('dataSource').to_frame('incident_records'))
if 'approved' in df.columns or 'verified' in df.columns:
    status_fields = [c for c in ['approved','verified'] if c in df.columns]
    display(df[status_fields].value_counts(dropna=False).rename('records').to_frame())

## 4. Save the processed snapshot and quality report

The processed table keeps the source columns and adds normalized fields and quality flags. It preserves all non-duplicate-ID rows, including rows with missing or out-of-bounds coordinates. This is a reproducible preparation artifact, not a claim that each observation is valid or that the source archive is complete.

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed' / 'bipad_incidents' / RUN_DIR.name
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
processed_path = OUTPUT_DIR / 'incidents_processed.csv'
report_path = OUTPUT_DIR / 'data_quality_report.json'
field_dictionary_path = OUTPUT_DIR / 'field_dictionary.csv'
df.to_csv(processed_path, index=False, encoding='utf-8-sig')
report = {'project':'UrbanPulse Nepal', 'source_endpoint':manifest.get('endpoint'), 'source_snapshot':RUN_DIR.name, 'source_retrieved_at_utc':manifest.get('retrieved_at_utc'), 'generated_at_utc':datetime.now(timezone.utc).isoformat(), 'rows':len(df), 'input_rows':len(raw_df), 'duplicate_ids_removed':duplicate_id_count, 'quality_audit':audit, 'coverage_limitation':'Pagination ended on an empty page, but API count was an implausible int64 maximum sentinel. Completeness of historical coverage is unverified.', 'hazard_label_limitation':'Hazard IDs are retained as IDs; readable labels require a verified BIPAD hazard lookup.', 'feature_definitions':{'longitude':'First GeoJSON point coordinate.', 'latitude':'Second GeoJSON point coordinate.', 'coordinates_present':'Both coordinates parse as finite numbers.', 'coordinates_in_broad_nepal_bounds':'Finite point with latitude 26–31 and longitude 80–89; broad sanity check only.', 'incident_year':'Year of incidentOn after conversion to Asia/Kathmandu.', 'incident_month':'Month of incidentOn after conversion to Asia/Kathmandu.', 'incident_season':'Nepal calendar month grouped into Winter (Dec–Feb), Pre-monsoon (Mar–May), Monsoon (Jun–Sep), Post-monsoon (Oct–Nov).'}}
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
field_dictionary = pd.DataFrame([{'field':'longitude','definition':'First coordinate from GeoJSON point.coordinates; longitude order verified from sample records.'},{'field':'latitude','definition':'Second coordinate from GeoJSON point.coordinates.'},{'field':'coordinates_present','definition':'Both coordinates parsed as finite numeric values.'},{'field':'coordinates_in_broad_nepal_bounds','definition':'Latitude is 26–31 and longitude is 80–89; broad check only.'},{'field':'hazard_id','definition':'Original BIPAD hazard identifier stored as a string; no label inferred.'},{'field':'incident_year','definition':'Year from incidentOn converted to Asia/Kathmandu.'},{'field':'incident_month','definition':'Month from incidentOn converted to Asia/Kathmandu.'},{'field':'incident_season','definition':'Winter Dec–Feb; Pre-monsoon Mar–May; Monsoon Jun–Sep; Post-monsoon Oct–Nov.'}])
field_dictionary.to_csv(field_dictionary_path, index=False, encoding='utf-8-sig')
print('Processed data:', processed_path)
print('Quality report:', report_path)
print('Feature dictionary:', field_dictionary_path)
display(pd.Series(audit, name='value').to_frame())
print('Share the executed notebook output and data_quality_report.json for the next modeling stage.')